# Análise da Tabela 3 — Áreas gerais de formação na graduação

**Fonte:** IBGE — Censo Demográfico (Tabela 10063 — Pessoas com nível superior completo, por áreas gerais de formação do curso de graduação concluído, segundo o sexo).

Objetivo: dividir a Tabela 3 em três tabelas, com **homens e mulheres por Grande Região**:

1. Com curso superior (total)
2. Com curso nas áreas de **Ciência, Tecnologia, Engenharias e Matemática**
3. Com curso nas áreas de **Educação, Serviços pessoais, Saúde e Bem-estar**

Cada tabela é salva em `.xlsx` e analisada com `describe()`.

In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Leitura da Tabela 3

In [2]:
ARQUIVO = 'Tabela_3_Areas_gerais_de_formacao_na_graduacao.xlsx'

# As 3 primeiras linhas são título e cabeçalhos mesclados; a última é a fonte
bruto = pd.read_excel(ARQUIVO, header=None)
bruto.head(5)

,0,1,2,3,4,5,6,7,8,9
0,Tabela 10063 - Pessoas com nível superior comp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Brasil, Grande Região e Unidade da Federação",Total,NaN,NaN,"Ciência, Tecnologia, Engenharias e Matemática",NaN,NaN,"Educação, Serviços pessoais, Saúde e Bem-estar",NaN,NaN
2,NaN,Total,Homens,Mulheres,Total,Homens,Mulheres,Total,Homens,Mulheres
3,Brasil,25854291,10478534,15375757,4149041,2750422,1398619,8079707,1677985,6401723
4,Norte,1525166,590558,934609,209657,133055,76601,599785,141169,458616


In [3]:
colunas = [
    'Local',
    'Total_Total', 'Total_Homens', 'Total_Mulheres',
    'CTEM_Total', 'CTEM_Homens', 'CTEM_Mulheres',
    'ESSB_Total', 'ESSB_Homens', 'ESSB_Mulheres',
]

dados = bruto.iloc[3:-1].copy()          # remove título, cabeçalhos e linha da fonte
dados.columns = colunas
dados['Local'] = dados['Local'].str.strip()
dados[colunas[1:]] = dados[colunas[1:]].astype(np.int64)
dados = dados.set_index('Local')
dados

,Total_Total,Total_Homens,Total_Mulheres,CTEM_Total,CTEM_Homens,CTEM_Mulheres,ESSB_Total,ESSB_Homens,ESSB_Mulheres
Local,,,,,,,,,
Brasil,25854291,10478534,15375757,4149041,2750422,1398619,8079707,1677985,6401723
Norte,1525166,590558,934609,209657,133055,76601,599785,141169,458616
Nordeste,4748933,1744994,3003939,621388,389993,231395,1822087,363905,1458181
Sudeste,12828829,5391638,7437191,2328122,1588874,739247,3652397,768288,2884108
Sul,4338861,1768855,2570005,662916,427372,235543,1256291,248574,1007718
Centro-Oeste,2412502,982489,1430013,326959,211124,115834,749148,156048,593101
Rondônia,164632,62766,101866,20538,11863,8675,63584,13963,49622
Acre,84300,33291,51009,10364,6237,4128,33379,9121,24258
Amazonas,342566,141059,201507,52669,34749,17919,121988,30738,91249


## 2. Filtrando as Grandes Regiões

A tabela traz Brasil, as 5 Grandes Regiões e as 27 UFs. Para a análise **por região** ficamos só com as 5 regiões (o total Brasil distorceria as estatísticas).

In [4]:
regioes = ['Norte', 'Nordeste', 'Sudeste', 'Sul', 'Centro-Oeste']
por_regiao = dados.loc[regioes]

# Conferência: a soma das regiões deve bater com o total Brasil (diferenças de arredondamento do IBGE são esperadas)
diferenca = por_regiao.sum() - dados.loc['Brasil']
diferenca

Total_Total       0
Total_Homens      0
Total_Mulheres    0
CTEM_Total        1
CTEM_Homens      -4
CTEM_Mulheres     1
ESSB_Total        1
ESSB_Homens      -1
ESSB_Mulheres     1
dtype: int64

## 3. Divisão em 3 tabelas

In [5]:
def monta_tabela(prefixo):
    t = por_regiao[[f'{prefixo}_Homens', f'{prefixo}_Mulheres']].copy()
    t.columns = ['Homens', 'Mulheres']
    t.index.name = 'Região'
    return t

superior   = monta_tabela('Total')   # 1. curso superior (todas as áreas)
tecnologia = monta_tabela('CTEM')    # 2. Ciência, Tecnologia, Engenharias e Matemática
educacao   = monta_tabela('ESSB')    # 3. Educação, Serviços pessoais, Saúde e Bem-estar

### 3.1 Homens e mulheres por região com curso superior

In [6]:
superior

,Homens,Mulheres
Região,,
Norte,590558,934609
Nordeste,1744994,3003939
Sudeste,5391638,7437191
Sul,1768855,2570005
Centro-Oeste,982489,1430013


### 3.2 Homens e mulheres por região — Ciência, Tecnologia, Engenharias e Matemática

In [7]:
tecnologia

,Homens,Mulheres
Região,,
Norte,133055,76601
Nordeste,389993,231395
Sudeste,1588874,739247
Sul,427372,235543
Centro-Oeste,211124,115834


### 3.3 Homens e mulheres por região — Educação, Serviços pessoais, Saúde e Bem-estar

In [8]:
educacao

,Homens,Mulheres
Região,,
Norte,141169,458616
Nordeste,363905,1458181
Sudeste,768288,2884108
Sul,248574,1007718
Centro-Oeste,156048,593101


## 4. Salvando as tabelas em .xlsx

In [9]:
PASTA = 'tabelas'
os.makedirs(PASTA, exist_ok=True)

superior.to_excel(os.path.join(PASTA, 'superior.xlsx'))
tecnologia.to_excel(os.path.join(PASTA, 'tecnologia.xlsx'))
educacao.to_excel(os.path.join(PASTA, 'educacao.xlsx'))

os.listdir(PASTA)

['educacao.xlsx', 'superior.xlsx', 'tecnologia.xlsx']

## 5. Estatísticas descritivas — `describe()`

### 5.1 Curso superior

In [10]:
superior.describe()

,Homens,Mulheres
count,5.00,5.00
mean,"2,095,706.80","3,075,151.40"
std,"1,910,348.19","2,577,597.51"
min,"590,558.00","934,609.00"
25%,"982,489.00","1,430,013.00"
50%,"1,744,994.00","2,570,005.00"
75%,"1,768,855.00","3,003,939.00"
max,"5,391,638.00","7,437,191.00"


### 5.2 Ciência, Tecnologia, Engenharias e Matemática

In [11]:
tecnologia.describe()

,Homens,Mulheres
count,5.00,5.00
mean,"550,083.60","279,724.00"
std,"593,417.87","266,255.48"
min,"133,055.00","76,601.00"
25%,"211,124.00","115,834.00"
50%,"389,993.00","231,395.00"
75%,"427,372.00","235,543.00"
max,"1,588,874.00","739,247.00"


### 5.3 Educação, Serviços pessoais, Saúde e Bem-estar

In [12]:
educacao.describe()

,Homens,Mulheres
count,5.00,5.00
mean,"335,596.80","1,280,344.80"
std,"257,699.41","977,950.63"
min,"141,169.00","458,616.00"
25%,"156,048.00","593,101.00"
50%,"248,574.00","1,007,718.00"
75%,"363,905.00","1,458,181.00"
max,"768,288.00","2,884,108.00"


## 6. Comparação com NumPy — participação feminina por região

In [13]:
participacao = pd.DataFrame({
    'Superior (%)':   np.round(superior['Mulheres']   / superior.sum(axis=1)   * 100, 2),
    'CTEM (%)':       np.round(tecnologia['Mulheres'] / tecnologia.sum(axis=1) * 100, 2),
    'Educ./Saúde (%)': np.round(educacao['Mulheres']  / educacao.sum(axis=1)   * 100, 2),
})
participacao

,Superior (%),CTEM (%),Educ./Saúde (%)
Região,,,
Norte,61.28,36.54,76.46
Nordeste,63.26,37.24,80.03
Sudeste,57.97,31.75,78.96
Sul,59.23,35.53,80.21
Centro-Oeste,59.28,35.43,79.17


In [14]:
razao = pd.DataFrame({
    'Superior':    np.round(superior['Mulheres']   / superior['Homens'], 2),
    'CTEM':        np.round(tecnologia['Mulheres'] / tecnologia['Homens'], 2),
    'Educ./Saúde': np.round(educacao['Mulheres']   / educacao['Homens'], 2),
})
razao  # mulheres para cada homem

,Superior,CTEM,Educ./Saúde
Região,,,
Norte,1.58,0.58,3.25
Nordeste,1.72,0.59,4.01
Sudeste,1.38,0.47,3.75
Sul,1.45,0.55,4.05
Centro-Oeste,1.46,0.55,3.80


## 7. Conclusões

- **Curso superior:** em todas as regiões há mais mulheres do que homens formados (cerca de 6 em cada 10 graduados são mulheres).
- **CTEM:** o quadro se inverte — os homens são maioria em todas as regiões, com as mulheres representando apenas cerca de um terço dos formados.
- **Educação, Serviços pessoais, Saúde e Bem-estar:** forte predominância feminina, em torno de 3 a 4 mulheres para cada homem.
- O **Sudeste** concentra os maiores valores absolutos em todas as tabelas, o que explica o desvio padrão elevado e a média bem acima da mediana no `describe()`.